# 02.3 Broadcasting: The Rules, and the Shapes That Lie

> **Prerequisites:** 02.1 (strides and views — the stretch below is a stride trick) ·
> 02.2 (dtypes; the INR lane and its boundary parsing carry over)
> **What you'll learn:**
> - Apply the broadcasting rule by hand — align right, pair with equal-or-1, stretch the 1s — before numpy applies it for you
> - Tell the three fan-outs apart: intended elementwise, declared outer, and accidental outer — and write the intent into the code
> - Predict an operation's result *shape and memory* from the operand shapes alone, before allocating anything
> - Recognise the `(n,) − (n,1)` accident, why it runs clean at batch scale and dies at full scale, and what it reports in between
> - Replace per-row Python loops with broadcast expressions, and know what that buys and costs
> **Level:** Beginner · **Series:** 02 NumPy & Vectorized Computing

> ⚡ **Tuesday 2026-09-01, 02:10** — the month-end anomaly job for August finishes green and
> pages the entire on-call rota: the deviation metric has jumped 42-fold and every INR
> customer is flagged anomalous at once. Memory on the batch host blipped by 70 MB that
> nobody noticed. The cause: two weeks ago, a helper function started returning its answer
> as a column.


## Concept
### Plain-English Explanation

Broadcasting is numpy's rule for doing arithmetic between arrays of different shapes
without writing loops: a scalar meets an array and applies to every element; a row of four
thresholds meets a column of invoices and produces every combination. It is the mechanism
that makes vectorized code short, and 02.1 already showed its price tag is small — the
"stretching" is a stride trick, not a copy.

The catch is that broadcasting never announces itself. The same silent rule that stretches
a scalar across a column will just as happily stretch two *almost-matching* arrays into a
grid of every pairwise combination — and then hand that grid to whatever reduction comes
next, which will dutifully average a million numbers you never meant to create. Nothing
raises, because nothing is wrong by the rule's lights. The cold open is exactly this: a
metric that should have averaged 2,945 deviations averaged 8,670,080 extra cross-pairs
besides, because one operand had picked up a second dimension of length one.

### Technical Explanation

The rule, in full — there is no more to it than this. To evaluate an elementwise operation
on two arrays, numpy **aligns their shapes at the right edge**, padding the shorter shape
with 1s on the left. Each aligned pair of dimensions must be **equal, or contain a 1**; a 1
is *stretched* to match its partner — implemented as a stride-0 axis over the same buffer,
so the stretch re-reads bytes rather than copying them (02.1's header trick, applied). Any
other pairing refuses with an error. The result's shape takes the maximum of each pair.

⭐ **CRITICAL CONCEPT** — right-alignment is where intuitions break. `(2, 3) + (3,)` works,
because the 3s meet at the right edge; `(2, 3) + (2,)` *refuses*, because 2 meets 3 — even
though a human squinting at it can see a "compatible" left edge. And the incident's shape
pair is legal precisely because of the padding step: `(n,) − (n, 1)` right-aligns to
`(1, n) − (n, 1)`, both 1s stretch, and the result is `(n, n)` — every element of one
operand against every element of the other. Legal, silent, and almost never what an
elementwise-looking line of code meant.

The numbers that matter here: the August 2026 INR batch is 2,945 invoices from a lane of
116,888 across 3,265 customers. Elementwise deviation from each invoice's own customer mean
is 2,945 float64 values — effectively free. The accidental outer version is a
2,945 × 2,945 matrix: 69.4 MB, allocated and reduced without complaint. The same accident
on the full lane asks for 102 GiB and dies loudly — the *batch size*, not the bug, decides
whether you get a wrong number or a crash.

### Mental Model

Broadcasting is a shape *proof*, not a shape *guess*: align right, pair each dimension with
an equal or a 1, stretch the 1s for free. Run the proof in your head before numpy runs it
for real — the result shape and its byte count are known before a single element exists,
and any shape you did not predict is a bug you have not met yet.


## How It Works

```text
  THE RULE: align RIGHT; pad the shorter shape with 1s; each pair equal-or-1;
            every 1 stretches by stride 0 (re-read, never copied); else REFUSE.

  the incident's pair:            a   (2945,)   ->  (   1, 2945)
                                  m   (2945,1)  ->  (2945,    1)
                                  result            (2945, 2945)   69.4 MB

  every invoice amount  a_j  meets every customer-mean  m_i:  out[i, j] = a_j - m_i
  the 2,945 numbers the code MEANT live on the diagonal  (i == j);
  the other 8,670,080 entries are cross-pairs no one asked about - and .mean()
  averages all of them without comment.

  three fan-outs, one mechanism:
    intended elementwise   (n,)  - (n,)     n values        the metric
    declared outer         (n,1) >= (k,)    n*k, reduced    the bucket table, on purpose
    accidental outer       (n,)  - (n,1)    n*n, REPORTED   the incident
```

Read the diagram's middle block twice, because it contains the incident's cruelest
property: **the correct answer is inside the wrong matrix.** Element `[i, j]` of the
accidental grid is `a[j] − m[i]` — a real difference of real numbers — so its diagonal,
where `i == j`, is exactly the elementwise result the code meant to compute. Every entry is
plausible; the matrix as a whole is meaningless; and the reduction that follows launders
the mistake into a single confident scalar. There is no NaN, no overflow, no warning
anywhere in that pipeline — shape was the only witness, and nothing was checking shape.

The bottom block is the honest taxonomy. The same mechanism serves three purposes
distinguished only by *intent*: elementwise arithmetic between identically-shaped arrays;
deliberate outer products, written with an explicit `np.newaxis` and reduced immediately;
and the accident, which is a declared-outer's shape reached by an undeclared route. The
whole discipline of this notebook is making the middle column's explicitness mandatory, so
the third column cannot impersonate the first.


## Hands-On Build
### Stage A — from scratch

The rule fits in fifteen lines, and implementing it once inoculates better than memorising
it. Below: the shape algebra by hand, and an explicit-loop outer computation that shows
what the stretched arithmetic *means* — both parity-checked against numpy, on real invoice
amounts. (The committed lab carries the same builds standalone.)


In [1]:
# Load the committed lab module; it owns the INR-lane parse (M7 commas at the
# boundary, per-currency by construction) and every experiment this notebook shows.
import importlib.util
import sys
from pathlib import Path

import numpy as np

LAB = Path.cwd() / "_lab" / "lab_02.3_broadcasting.py"
spec = importlib.util.spec_from_file_location("lab_02_3", LAB)
lab = importlib.util.module_from_spec(spec)
sys.modules["lab_02_3"] = lab
spec.loader.exec_module(lab)

amounts, cust, in_batch = lab.load_inr_lane()
means = lab.customer_means(amounts, cust)
print(f"INR lane: {len(amounts):,} invoices, {len(np.unique(cust)):,} customers; "
      f"August-2026 batch: {int(in_batch.sum()):,}")
print(f"per-invoice customer mean: shape {means.shape} - built with numpy alone:")
print( "  unique(return_inverse) labels rows, bincount(weights=) scatter-adds,")
print( "  means[inverse] gathers back to invoice order (02.4 owns that gather)")

INR lane: 116,888 invoices, 3,265 customers; August-2026 batch: 2,945
per-invoice customer mean: shape (116888,) - built with numpy alone:
  unique(return_inverse) labels rows, bincount(weights=) scatter-adds,
  means[inverse] gathers back to invoice order (02.4 owns that gather)


Those three lines — `unique`, `bincount`, a fancy-index gather — are a groupby with no
library on top, and they set up everything below: `means` is aligned one-to-one with
`amounts`, shape `(116888,)`, so the deviation the anomaly job wants is a plain elementwise
subtraction. Hold that "one-to-one" property; the incident is what happens when a refactor
quietly breaks it.

Now the rule itself, by hand.


In [2]:
def my_broadcast_shape(s1: tuple, s2: tuple) -> tuple:
    """The ENTIRE broadcasting rule: align right, pad with 1s, pair equal-or-1,
    stretch the 1s, refuse anything else. numpy does nothing more than this."""
    out = []
    for i in range(1, max(len(s1), len(s2)) + 1):
        d1 = s1[-i] if i <= len(s1) else 1          # pad missing LEFT dims with 1
        d2 = s2[-i] if i <= len(s2) else 1
        if d1 != 1 and d2 != 1 and d1 != d2:
            raise ValueError(f"incompatible: {s1} vs {s2} at axis -{i} ({d1} vs {d2})")
        out.append(max(d1, d2))                     # the 1 stretches to its partner
    return tuple(reversed(out))


def outer_add_loops(col: np.ndarray, row: np.ndarray) -> np.ndarray:
    """What (k,1) + (1,m) MEANS, spelled as loops: every pair, 1s re-reading."""
    out = np.empty((len(col), len(row)))
    for i in range(len(col)):
        for j in range(len(row)):
            out[i, j] = col[i] + row[j]
    return out

In [3]:
cases = [((7,), (7,)), ((7,), (1,)), ((7,), (7, 1)), ((3, 1), (1, 4)),
         ((2, 3), (3,)), ((5, 1, 4), (7, 1))]
print(f"{'shapes':<24}{'my rule':>14}{'numpy':>14}")
for s1, s2 in cases:
    mine, ref = my_broadcast_shape(s1, s2), np.broadcast_shapes(s1, s2)
    assert mine == ref
    print(f"{f'{s1} + {s2}':<24}{str(mine):>14}{str(ref):>14}   MATCH")

for fn, who in ((my_broadcast_shape, "my rule"), (np.broadcast_shapes, "numpy")):
    try:
        fn((2, 3), (2,))
    except ValueError:
        print(f"(2, 3) + (2,)           -> {who} refuses (right-aligned, 2 meets 3)")

hand = outer_add_loops(amounts[:3], lab.BUCKETS_INR)
bcast = amounts[:3].reshape(3, 1) + lab.BUCKETS_INR.reshape(1, 4)
print(f"\nouter add, 3 invoices x 4 thresholds: loops == broadcast -> "
      f"{np.array_equal(hand, bcast)}  (shape {bcast.shape})")

shapes                         my rule         numpy
(7,) + (7,)                       (7,)          (7,)   MATCH
(7,) + (1,)                       (7,)          (7,)   MATCH
(7,) + (7, 1)                   (7, 7)        (7, 7)   MATCH
(3, 1) + (1, 4)                 (3, 4)        (3, 4)   MATCH
(2, 3) + (3,)                   (2, 3)        (2, 3)   MATCH
(5, 1, 4) + (7, 1)           (5, 7, 4)     (5, 7, 4)   MATCH
(2, 3) + (2,)           -> my rule refuses (right-aligned, 2 meets 3)
(2, 3) + (2,)           -> numpy refuses (right-aligned, 2 meets 3)

outer add, 3 invoices x 4 thresholds: loops == broadcast -> True  (shape (3, 4))


Six MATCH lines, one refusal, and a loop parity — the whole mechanism, verified. Note
which cases the table settles: `(7,) + (7, 1)` produces `(7, 7)` — the incident's pair, in
miniature, blessed by the rule; and `(2, 3) + (2,)` refuses even though it "looks"
alignable, because the rule only ever reads shapes from the right. The stretched-1
arithmetic *means* the double loop above it, nothing more mysterious — numpy just runs that
loop in C over a stride-0 axis.

### Stage B — idiomatic

Two measurements before the incident: what broadcasting buys over the loops it replaces,
and what the stretch costs in memory (nothing — which is also *why* the accident allocates
so eagerly at the next step).


In [4]:
lab.loops_vs_broadcast(amounts)

  bucket counts agree: [49702, 36051, 13270, 1424]
  python loops :    122.7 ms
  broadcast    :      3.3 ms   (~37x on this run - machine-dependent, read as an order of magnitude)


Same counts, an order of magnitude apart on this machine (the exact multiple moves run
to run — 01.4's benchmark honesty applies; rerun the cell and watch it wobble). The
broadcast spelling `a[:, np.newaxis] >= thresholds` is doing the double loop in compiled
code, and the `np.newaxis` is load-bearing twice over: it makes the fan-out *happen*, and it
makes the fan-out *visible to the reader* — this expression could never be mistaken for
elementwise arithmetic. That visibility is the entire Stage C argument, made early.

Where does the stretched operand's memory go? Nowhere — and proving it ties this notebook
back to 02.1's header arithmetic.


In [5]:
lab.stride_zero(amounts)

  np.broadcast_to(thresholds, (10,000, 4)):
    shape (10000, 4)   strides (0, 8)   base buffer 32 bytes
    the first stride is 0: moving 'down' re-reads the SAME 32 bytes - the
    stretch is 02.1's header trick, not an allocation. Materialise it
    (np.ascontiguousarray) and it would cost 320 KB for real.
    writeable: False - numpy refuses writes through a
    stretched view, because one write would 'land' in 10,000 places at once


A stride of 0 on the stretched axis: moving "down" ten thousand rows re-reads the same
32-byte threshold buffer, so `broadcast_to` allocates nothing no matter how far it
stretches. Two consequences worth keeping. First, broadcasting's *inputs* are free — the
cost appears only when an operation **materialises a result** at the broadcast shape, which
is why the incident's subtraction, not its operands, allocated 69.4 MB. Second, numpy marks
the stretched view unwritable: one write would land in ten thousand logical places at once,
so the refusal is 02.1's read-only guard arriving built-in.

The legitimate version of the incident's fan-out, before the incident itself:


In [6]:
lab.size_buckets(amounts, in_batch)

  a[:, np.newaxis] >= thresholds: (2945,) x (4,) -> (2945, 4)  (12 KB of bool)
    >= INR       1,000 :  2,945 invoices
    >= INR      10,000 :  2,168 invoices
    >= INR     100,000 :    764 invoices
    >= INR   1,000,000 :     91 invoices
  the same fan-out mechanism as the incident - declared with np.newaxis,
  sized k=4 by design, and reduced immediately. Intent, written down.


This is the declared outer from the taxonomy: `(n, 1)` against `(4,)`, fanning out to
an `(n, 4)` boolean grid — 12 KB, sized by design because `k` is four thresholds and not a
second copy of `n` — and reduced immediately by `sum(axis=0)` into the bucket counts. Same
mechanism as the accident; opposite in every property that matters: the extra axis is
written explicitly (`np.newaxis`), the fanned dimension is small by construction, and the
grid never leaves the expression that reduces it.

### Stage C — production

The accident's whole trick is impersonating elementwise arithmetic. The production answer
makes the impersonation impossible at the call sites that mean "element by element": an
elementwise-or-die wrapper, and a one-line preflight that prices any intended fan-out
before allocating it.


In [7]:
lab.contract_demo(amounts, cust, in_batch)

  checked_elementwise(a (n,), m (n,))     -> ok, shape (2945,)
  checked_elementwise(a (n,), m (n,1))    -> ShapeError: result (2945, 2945) != operand shapes (2945,)/(2945, 1): an implicit broadcast is about to fan out

  preflight pricing, before any allocation:
    (n,) - (n,)   : (2945,) -> 0.0 MB
    (n,) - (n,1)  : (2945, 2945) -> 69.4 MB
    full lane bug : (116888, 116888) -> 109,302.4 MB
  the guard turns the silent batch-size lottery into a refusal; the
  preflight turns the loud MemoryError into a line in a design review


`checked_elementwise` inverts numpy's default: broadcasting becomes *opt-in*. The
correct `(n,) − (n,)` passes through untouched; the incident's `(n,) − (n, 1)` raises
`ShapeError` naming the fan-out it was about to perform — at the offending line, before any
allocation, which is 02.1's detection-distance-zero property again. Call sites that
genuinely mean an outer computation skip the wrapper and say so with `np.newaxis`, so
reading the code now distinguishes the taxonomy's three columns at a glance.

The preflight is the other half: `np.broadcast_shapes` runs the shape proof without
touching data, so the price of any proposed operation — 0.0 MB elementwise, 69.4 MB for the
batch accident, 109,302.4 MB for the full-lane one — is a printable fact *before* the
allocator finds out. A design review that quotes that line cannot ship the crash version by
surprise.

## Evaluation

Assertion-shaped, as this series' toolkit notebooks are (guide §2). Three captured layers:
the **Stage A parity table** — six shape cases plus the refusal, asserted equal between the
hand rule and `np.broadcast_shapes`, plus the loop-vs-broadcast outer equality — which is
the mechanism's regression test; the **bucket-count equality** inside the timing cell,
asserting the vectorized rewrite computes exactly what the loops computed before any
speedup is claimed; and the **contract pair**, where the correct shape passes and the
incident's shape raises. Everything is deterministic; a changed line is a changed behaviour,
not noise.


## Design Patterns / Tradeoffs

**Implicit broadcasting everywhere versus elementwise-or-die at boundaries.** numpy's
default — every operation broadcasts silently — is what makes idiomatic code terse, and
inside a ten-line function whose shapes are all local it is the right default: guards on
every line would drown the arithmetic. The failure mode is exactly the incident: at
*boundaries* — metric functions, anything consuming shapes it did not construct — the same
silence lets an aliased-in `(n, 1)` rewrite the computation. Keep implicit broadcasting
inside tight scopes; put `checked_elementwise` (or an explicit
`assert a.shape == b.shape`) at the seams where arrays arrive from elsewhere. The wrapper
costs one shape proof per call — microseconds — and converts a batch-size lottery into a
stack trace.

**`np.newaxis` versus `reshape(-1, 1)` for declaring a fan-out.** Both produce the same
`(n, 1)` view; the difference is what they say to the reader and where they sit.
`a[:, np.newaxis]` lives *inside the expression that fans out*, so intent and mechanism are
one line; `reshape(-1, 1)` detaches the shape change from its consumer — the incident's
helper reshaped "for a later step", and the shape travelled to a subtraction that never
wanted it. Prefer `np.newaxis` at the point of use; treat a `reshape(-1, 1)` in returned
values as a smell — return `(n,)` and let consumers add axes they need, where they need
them.

**Broadcast versus chunked loop when the fanned dimension is large.** The bucket grid was
`n × 4`; make it `n × k` with both large — every invoice against every calendar day, say —
and the intermediate outgrows RAM even when the *reduced* answer is tiny. The preflight
decides: price `n·k·itemsize`; past a few hundred MB, chunk the outer loop over one axis
and broadcast within chunks, trading one Python-level loop for bounded memory. (02.7 adds
the einsum/`np.add.at` family for reductions that never materialise the grid at all.)

**Recommendation for PayFlow:** metric and reconciliation call sites use
`checked_elementwise`; deliberate outers write `np.newaxis` at the point of use and reduce
in the same expression; any broadcast whose fanned product can exceed ~100 MB quotes its
preflight line in review; helpers return `(n,)` unless their name says otherwise.


## Production Scenario
### Symptoms

**Tuesday 2026-09-01, 02:10.** The month-end anomaly job computes, per INR invoice, the
absolute deviation from that customer's mean amount, and alerts when the batch's average
deviation crosses a threshold calibrated in 2025.

- **02:10** — the job completes successfully and fires its alert — and the alert's payload
  is absurd: the August metric is 42 times its usual level, and at that level *every*
  customer's invoices read as anomalous. The rota treats it as a data incident.
- The batch host's memory monitor shows a 70 MB allocation spike during the run — noted in
  passing, alarming no one; the box has headroom.
- Job logs are clean: no exception, no warning, wall-clock within its normal band.
- The August exports pass 01.2's contract checks; row count for the batch (2,945 INR
  invoices) is in line with July's. Nothing upstream moved.
- The metric's history shows the jump began exactly at the previous month-end run — the
  first run after a refactor PR titled "return customer means as a column for the upcoming
  matrix pipeline".


In [8]:
summary = lab.incident(amounts, cust, in_batch)

  August 2026 INR batch: n=2,945 invoices
  correct: (n,) - (n,)     -> shape (2945,), 0.0 MB   metric = 4,674.16
  shipped: (n,) - (n,1)    -> shape (2945, 2945), 69.4 MB   metric = 197,560.13
  the job RAN: the intermediate fits in RAM at this n, every value in it is
  a real difference of real numbers - invoice i minus customer-mean j, for
  ALL pairs - and the metric inflated 42.3x
  the diagonal of the (n,n) mistake IS the correct answer: 4,674.16 -
  the right 2,945 numbers, drowned among 8,670,080 meaningless pairs

  the same bug on the full INR lane (116,888 rows):
    MemoryError: Unable to allocate 102. GiB for an array with shape (116888, 11688
    loud at scale, silent in the batch - the batch size decided the
    symptom, not the bug


### Diagnosis

Walking the ladder in its numerical-incident form, naming what each signal eliminated:

1. **Alert** — a 42-fold metric jump with a green job. Candidate causes: the data moved, the
   metric's inputs moved, or the metric's *computation* moved.
2. **Input checks** — exports hash clean, contract checks pass, batch size normal. Data
   eliminated; and since per-customer means over the same lane are unchanged, the inputs to
   the metric are individually healthy.
3. **Job logs and telemetry** — the logs are clean, and the 70 MB spike in the memory
   dashboard is the tell nobody read: the metric's working set
   should be one float64 vector, a fraction of a megabyte. A memory footprint three orders
   above the data size means an *intermediate* exists that the design never drew.
4. **Recompute stepwise, printing shapes** — the cell above: the deviation array is
   `(2945, 2945)`, not `(2945,)`. The operands are `(n,)` and `(n, 1)`; the subtraction is
   an outer difference; the mean then averages the full grid — the 2,945 intended
   values drowned among 8,670,080 cross-pairs. Confirmed mechanically: the diagonal's mean is 4,674.16 —
   the correct August metric — while the reported figure is 197,560.13.
5. **Version diff** — no library moved; the one code change in the window is the
   month-old refactor: `customer_means()` gained a
   `reshape(-1, 1)` on its return "to align with a planned matrix step". Every consumer
   inherited a column where a vector had been promised.
6. **Mechanism named** — `(n,) − (n, 1)` right-aligns to `(1, n) − (n, 1)`; both 1s
   stretch; the rule is satisfied and silent. The batch fit in RAM, so the job ran; the
   same expression on the full 116,888-row lane refuses with `MemoryError: Unable to
   allocate 102. GiB` — the bug's *symptom* was chosen by the batch size, not by the code.

### Root Cause

A helper began returning per-customer means shaped `(n, 1)` instead of `(n,)`, and the
anomaly metric's elementwise subtraction silently broadcast to an `(n, n)` outer
difference; the downstream mean then reported a cross-pair statistic 42 times the true
deviation metric, with no error raised at any step because every step was individually
legal.

### Fix

**Mitigation now.** Ravel at the metric's call site (`means.ravel()`), rerun August — the
corrected metric is 4,674.16, comfortably inside its threshold — and retract the alert
storm with the one-line explanation the diagonal provides: the right numbers were on the
diagonal of the wrong matrix.

**Permanent fix.** The helper returns `(n,)` again — consumers that need a column add
`np.newaxis` at their own point of use — and the metric path adopts Stage C:
`checked_elementwise` at the seam, so a reshaped operand raises `ShapeError` at the
offending line instead of inflating a statistic two steps later.

### Prevention

- **Shape contracts at metric seams.** One shape proof per call; the incident's exact
  operands are the wrapper's captured refusal case.
- **Preflight any intended fan-out in review**: `np.broadcast_shapes` plus an itemsize
  multiply prices the intermediate before it exists — 69.4 MB would have been a question,
  109 GB a veto.
- **Memory telemetry gets a per-job baseline.** A metric job allocating three thousand
  times its own data size is a shape bug wearing a resource graph; the 70 MB spike said so a month early.
- **Helpers return `(n,)`.** Axis insertion belongs at the expression that fans out —
  `reshape(-1, 1)` "for later" hands a loaded shape to every caller in between.


## Common Pitfalls

⚠️ **`(n,)` meets `(n, 1)`.** The signature accident: legal by right-alignment, silent,
`(n, n)` result, and every entry a plausible number. If an "elementwise" line can receive
shapes it did not build, prove them first.

⚠️ **Reading "it ran" as "the shapes were right".** The batch accident allocated 69.4 MB
and reported a number; the full-lane accident crashed. Whether you get a wrong statistic or
a MemoryError is decided by `n` and your RAM — the bug is identical. A crash is the *lucky*
outcome.

**Expecting left-alignment.** `(2, 3) + (2,)` refuses — the rule reads shapes from the
right, always. When a "should work" pairing refuses, write both shapes right-aligned on
paper before reaching for `reshape`.

**`reshape(-1, 1)` in a return value.** A shape change detached from the expression that
needs it is a time bomb for every consumer expecting `(n,)` — declare axes with
`np.newaxis` at the point of use instead.

**Trusting a reduction to sanity-check its input.** `.mean()` averaged the 8,670,080 extra cross-pairs as
happily as the 2,945 intended values — reductions launder shape mistakes into confident
scalars. The check
belongs on the operand shapes, not the output's plausibility.

**Fanning out a large × large pair because the reduced answer is small.** The intermediate
is what allocates. Price `n·k·itemsize` first; chunk or restructure (02.7) past your memory
budget.

**Writing through a `broadcast_to` view.** numpy refuses — the stretched axis re-reads one
buffer, so a write would land everywhere at once. The refusal is a feature; wanting the
write means you wanted a materialised copy.


## Interview Questions

1. **Derive this.** State the broadcasting rule precisely, then use it to derive the result
   shape of `(n,) − (n, 1)` and of `(2, 3) + (2,)`. *Answer shape:* align right, pad left
   with 1s, each pair equal-or-1 with 1s stretching, else refuse; `(n,)` pads to `(1, n)`,
   both 1s stretch against their partners giving `(n, n)`; in the second case 3 meets 2 at
   the rightmost axis and the operation refuses — right-alignment, not plausibility, decides.
2. **Design this.** A metrics library consumes arrays it did not construct. Design its
   shape-safety policy. *Answer shape:* elementwise-or-die wrappers (broadcast opt-in) at
   the public seams; internal code broadcasts freely within local scope; deliberate outers
   written with `np.newaxis` at point of use and reduced in-expression; helpers return
   `(n,)`; intermediates priced by `np.broadcast_shapes` preflight past a stated MB budget.
3. **Debug this.** A statistic jumps 40-fold; the job is green, inputs pass their
   contracts, and the host shows an unexplained memory spike. Walk it. *Answer shape:*
   the spike sizes the intermediate — compare to the data's own footprint; recompute the
   pipeline stepwise printing shapes; find the fan-out; diff for the shape change that fed
   it; confirm by recovering the true statistic (here, the diagonal) from the wrong
   intermediate.
4. Why does broadcasting cost no memory on its inputs, and where does the memory go
   instead? *Answer shape:* stretched axes are stride-0 views over the original buffer
   (02.1) — re-reads, not copies; cost appears when an operation materialises its *result*
   at the broadcast shape, so the output allocation is the entire bill.
5. Why is the diagonal of the accidental `(n, n)` difference exactly the intended answer,
   and why does that make the bug worse rather than better? *Answer shape:* entry `[i, j]`
   is `a[j] − m[i]`, so `i == j` recovers the aligned pairs — but it means every entry is a
   real, plausible number, so no value-level check can catch what only a shape-level check
   can see.
6. When do you *not* vectorize with broadcasting? *Answer shape:* when the fanned
   intermediate's `n·k·itemsize` outgrows the memory budget even though the reduced result
   is small — chunk one axis, or use reductions that never materialise the grid; also when
   `k` is tiny and the loop is clearer than the axis gymnastics.
7. What does `a[:, np.newaxis]` communicate that `a.reshape(-1, 1)` does not, given they
   produce the same view? *Answer shape:* placement — newaxis sits inside the fanning
   expression, marking the fan-out where it happens; a detached reshape ships a booby-trapped
   shape to consumers that expected `(n,)`, which is precisely the incident's mechanism.


## Key Takeaways

- The whole rule: align right, pad with 1s, pair equal-or-1, stretch 1s by stride 0, refuse
  the rest — fifteen lines you can (and did) implement from scratch.
- Run the shape proof before numpy does: result shape and byte count are computable from
  operand shapes alone, and `np.broadcast_shapes` prices any operation without touching data.
- `(n,) − (n, 1)` is the signature accident: legal, silent, `(n, n)`, every entry plausible,
  the true answer on the diagonal — and whether it lies or crashes depends only on `n`.
- Fan-outs come in three kinds — elementwise, declared, accidental — and `np.newaxis` at the
  point of use is what keeps the third from impersonating the first.
- Broadcasting's inputs are free (stride-0 stretches, unwritable by design); the bill is the
  materialised result, so budget the intermediate, not the operands.
- At boundaries, make broadcasting opt-in: a one-line shape proof (`checked_elementwise`)
  turns the batch-size lottery into a stack trace at the offending line.
- Helpers return `(n,)`; axis insertion happens in the expression that fans out — a
  `reshape(-1, 1)` "for later" is a loaded shape handed to every caller.
- A reduction will launder any shape mistake into one confident scalar; check shapes where
  they are made, because output plausibility checks nothing.


## Related

**Backward**

- **02.1 The ndarray Memory Model** — stride-0 stretching is the header trick again, and the
  unwritable broadcast view is the read-only guard arriving built-in.
- **02.2 Numerical Dtypes & Promotion** — the INR lane, its boundary parsing, and the
  per-currency discipline all carry over; promotion decides the result *dtype* while this
  notebook's rule decides the result *shape*.
- **01.2 First Contact with the PayFlow Data Universe** — the join fan-out (M13) is this
  incident's relational cousin: silent cardinality change, plausible values, wrong aggregate.

**Forward**

- **02.4 Indexing & Selection** — the `means[inverse]` gather used here becomes the subject:
  fancy indexing's powers and its assignment trap.
- **02.5 Ufuncs, Reductions & Missing Data** — the reductions that consumed the fanned grid,
  done properly: axis semantics, `keepdims`, and where `(n, 1)` shapes legitimately arise.
- **02.7 Memory Layout & Performance** — chunking and grid-free reductions for the
  large × large cases the preflight vetoes.
